# NurtureJoy — DistilBERT Safety Model (Lightweight Version)

This notebook is optimized for **Colab free tier** or weaker GPUs.

## Main changes vs heavier training
- trains on `nurturejoy_safety_small.csv`
- max length = **128**
- fp16 enabled when available
- batch size kept small
- gradient accumulation used to simulate larger batch
- early stopping enabled
- max epochs = **3**


In [ ]:
# If needed in Colab, uncomment:
# !pip -q install transformers datasets evaluate accelerate scikit-learn pandas torch safetensors

import os
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

In [ ]:
DATA_PATH = "nurturejoy_safety_small.csv"
MODEL_NAME = "distilbert-base-uncased"
OUTPUT_DIR = "nurturejoy_distilbert_safety_light"

assert os.path.exists(DATA_PATH), f"Missing {DATA_PATH}"

df = pd.read_csv(DATA_PATH)
assert "text" in df.columns and "unsafe" in df.columns, "Dataset must have text and unsafe columns"

df = df.dropna(subset=["text", "unsafe"]).copy()
df["unsafe"] = df["unsafe"].astype(int)

print(df.shape)
print(df["unsafe"].value_counts())

In [ ]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=42,
    stratify=df["unsafe"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["unsafe"]
)

print("Train:", train_df.shape, "Val:", val_df.shape, "Test:", test_df.shape)

In [ ]:
label_list = [0, 1]
id2label = {0: "SAFE", 1: "UNSAFE"}
label2id = {"SAFE": 0, "UNSAFE": 1}

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_ds = Dataset.from_pandas(train_df[["text", "unsafe"]].rename(columns={"unsafe": "label"}), preserve_index=False)
val_ds   = Dataset.from_pandas(val_df[["text", "unsafe"]].rename(columns={"unsafe": "label"}), preserve_index=False)
test_ds  = Dataset.from_pandas(test_df[["text", "unsafe"]].rename(columns={"unsafe": "label"}), preserve_index=False)

def tokenize_fn(batch):
    return tokenizer(batch["text"], truncation=True, max_length=128)

train_ds = train_ds.map(tokenize_fn, batched=True)
val_ds   = val_ds.map(tokenize_fn, batched=True)
test_ds  = test_ds.map(tokenize_fn, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array(label_list),
    y=train_df["unsafe"].values
)
class_weights = torch.tensor(class_weights, dtype=torch.float)
class_weights

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label=id2label,
    label2id=label2id
)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss_fct = torch.nn.CrossEntropyLoss(weight=class_weights.to(logits.device))
        loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_weighted": f1_score(labels, preds, average="weighted"),
        "f1_macro": f1_score(labels, preds, average="macro"),
    }

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    report_to="none",
    save_total_limit=2
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)]
)

trainer.train()

In [ ]:
pred = trainer.predict(test_ds)
y_true = pred.label_ids
y_pred = np.argmax(pred.predictions, axis=1)

print(classification_report(y_true, y_pred, target_names=["SAFE", "UNSAFE"]))
print("Confusion matrix:")
print(confusion_matrix(y_true, y_pred))

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Saved model to:", OUTPUT_DIR)